In [2]:
import os
import uuid
import wikipedia
import arxiv
from dotenv import load_dotenv

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.tools import create_retriever_tool
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langsmith import Client

# ==========================================
# 1. ENVIRONMENT & API SETUP
# ==========================================
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

# ==========================================
# 2. SECURITY & CONNECTION FIXES
# ==========================================
# Fix Wikipedia 403/JSONDecodeError by generating a unique User-Agent
unique_id = uuid.uuid4().hex[:8]
wikipedia.set_user_agent(f"GenAI_Agent_{unique_id}/1.0 (user_{unique_id}@example.com)")

# Fix Arxiv AttributeError and HTTP 301
# The new arxiv package natively defaults to HTTPS, so we just map the old method.
if hasattr(arxiv, "Client"):
    arxiv.Search.results = lambda self: arxiv.Client().results(self)
else:
    # Fallback for older arxiv versions
    arxiv.root_url = "https://export.arxiv.org/api/"

# ==========================================
# 3. BUILD THE TOOLS
# ==========================================
# Tool 1: LangSmith Retriever
gemini_embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
texts = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
db = Chroma.from_documents(texts, gemini_embeddings)
retriever = db.as_retriever()

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for Smith Documentation Retriever"
)

# Tool 2: Wikipedia
api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=1000)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

# Tool 3: Arxiv
arxiv_wrapper = ArxivAPIWrapper(top_k_results=5, doc_content_chars_max=1000)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

# Combine all tools
tools = [wiki, arxiv_tool, retriever_tool]

# ==========================================
# 4. BUILD THE AGENT
# ==========================================
# Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Pull Prompt
client = Client()
prompt = client.pull_prompt(
    "hwchase17/openai-functions-agent",
    dangerously_pull_public_prompt=True
)

# Create Agent and Executor
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# ==========================================
# 5. EXECUTE THE TASK
# ==========================================
print("\n=== STARTING AGENT EXECUTION ===\n")
response = agent_executor.invoke({
    "input": "Tell me about the history of Artificial Intelligence from Wikipedia, and also check Arxiv for any recent papers on it."
})

print("\n=== FINAL OUTPUT ===")
print(response["output"])


=== STARTING AGENT EXECUTION ===



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'History of Artificial Intelligence'}`


Page: History of artificial intelligence
Summary: The history of artificial intelligence (AI) began in antiquity, with myths, stories, and rumors of artificial beings endowed with intelligence by master craftsmen. The study of logic and formal reasoning from antiquity to the present led to the development of the programmable digital computer in the 1940s, a machine predicated on abstract mathematical reasoning. This device and the ideas behind it inspired scientists to begin discussing the possibility of building an electronic brain.
The field of AI research was founded at a workshop held on the campus of Dartmouth College in 1956. At the workshop, the first AI program, Logic Theorist, was presented by future Turing Awardee Allen Newell and future Nobel Laureate Herbert A. Simon, with help from J. C. Shaw. Attendees of the workshop b